In [ ]:
# =============================================================================
# Multi-Model Super-Resolution Evaluation on 5 Indian Areas — Kaggle Cell
# Models: Fine Tuned(DiffFuSR), Fine Tuned(Real ESRGAN), Frozen(Swin2SR), 
#         Frozen(Sen2SR), Frozen(OpenSR)
# Metrics: PSNR, SSIM, SAM, ERGAS, LPIPS
# =============================================================================

import os, sys, glob, types, subprocess, importlib.util

# -----------------------------------------------------------------------------
# 1. System Setup & Dependencies
# -----------------------------------------------------------------------------
print("Configuring environment and installing dependencies...")
subprocess.run(["apt-get", "update", "-qq"], check=False)
subprocess.run(["apt-get", "install", "-y", "-qq", "git-lfs"], check=False)
subprocess.run(["git", "lfs", "install"], check=False)

PACKAGES = [
    "pystac-client", "rasterio", "scikit-image", "pytorch-lightning",
    "basicsr", "realesrgan", "opencv-python-headless", "tifffile", "scipy",
    "pytorch-fid", "thop", "einops", "timm", "kornia", "lpips",
    "gdown", "h5py", "tqdm", "pyyaml", "natsort", "yapf", "onnxruntime-gpu"
]
subprocess.run([sys.executable, "-m", "pip", "install", "-q"] + PACKAGES, check=False)

try:
    import torchvision.transforms.functional_tensor  # noqa: F401
except ModuleNotFoundError:
    import torchvision.transforms.functional as _functional
    sys.modules["torchvision.transforms.functional_tensor"] = _functional

import torch
import numpy as np
import matplotlib.pyplot as plt
import rasterio
from rasterio.windows import Window
from PIL import Image
from skimage.metrics import peak_signal_noise_ratio as psnr
from skimage.metrics import structural_similarity as ssim
from pystac_client import Client
from pytorch_lightning import seed_everything
import lpips
import onnxruntime as ort

BASE_DIR = "/kaggle/working"
os.chdir(BASE_DIR)
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {DEVICE}")

# Initialize LPIPS metric
lpips_fn = lpips.LPIPS(net='vgg').to(DEVICE)

# -----------------------------------------------------------------------------
# 2. Model Initializations
# -----------------------------------------------------------------------------
models_dict = {}

# --- A. Fine Tuned(DiffFuSR) ---
print("Setting up DiffFuSR...")
while hasattr(torch.load, "__wrapped__"):
    torch.load = torch.load.__wrapped__
_orig_torch_load = torch.load
def _safe_torch_load(*args, **kwargs):
    kwargs["map_location"] = "cpu"
    if "weights_only" in kwargs:
        kwargs["weights_only"] = False
    return _orig_torch_load(*args, **kwargs)
_safe_torch_load.__wrapped__ = _orig_torch_load
torch.load = _safe_torch_load

def _load_source(name, path):
    spec = importlib.util.spec_from_file_location(name, path)
    mod = importlib.util.module_from_spec(spec)
    spec.loader.exec_module(mod)
    return mod

if "imp" not in sys.modules:
    imp_shim = types.ModuleType("imp")
    imp_shim.load_source = _load_source
    sys.modules["imp"] = imp_shim

REPO_DIR = os.path.join(BASE_DIR, "DiffFuSR")
if not os.path.exists(REPO_DIR):
    subprocess.run(["git", "clone", "https://github.com/NorskRegnesentral/DiffFuSR.git", REPO_DIR], check=True)
for p in [REPO_DIR, os.path.join(REPO_DIR, "litsr")]:
    if p not in sys.path: sys.path.insert(0, p)
if not os.path.exists(os.path.join(REPO_DIR, "logs")):
    subprocess.run(["git", "clone", "https://huggingface.co/NorskRegnesentralSTI/DiffFuSR", "_hf"], cwd=REPO_DIR, check=True)
    subprocess.run(["mv", "_hf/logs", "logs"], cwd=REPO_DIR, check=True)

import litsr.models
import litsr.archs
from litsr.utils import read_yaml
from litsr.utils.registry import ModelRegistry, ArchRegistry

def register_modules(search_dir):
    if not os.path.exists(search_dir): return
    for root, _, files in os.walk(search_dir):
        for f in files:
            if f.endswith(".py") and not f.startswith("__"):
                fpath = os.path.join(root, f)
                rel_path = os.path.relpath(fpath, REPO_DIR)
                mod_name = os.path.splitext(rel_path)[0].replace(os.sep, ".")
                try: importlib.import_module(mod_name)
                except Exception: pass
for d in ["archs", "models", "litsr/archs", "litsr/models"]:
    register_modules(os.path.join(REPO_DIR, d))

CKPT_PATH = os.path.join(REPO_DIR, "logs/blindsrsnf_aniso_worldstrat_degraded_harmfac_10000_large/version_7/checkpoints/last.ckpt")
if os.path.exists(CKPT_PATH):
    config = read_yaml(os.path.join(os.path.dirname(os.path.dirname(CKPT_PATH)), "hparams.yaml"))
    difffusr_model = litsr.models.load_model(config, CKPT_PATH, strict=False).to(DEVICE).eval()
    models_dict["Fine Tuned(DiffFuSR)"] = difffusr_model
else:
    print("DiffFuSR Checkpoint missing, skipping...")

# --- B. Fine Tuned(Real ESRGAN) ---
print("Setting up Real-ESRGAN...")
from basicsr.archs.rrdbnet_arch import RRDBNet
from realesrgan import RealESRGANer
rrdb_model = RRDBNet(num_in_ch=3, num_out_ch=3, num_feat=64, num_block=23, num_grow_ch=32, scale=4)
realesrgan_model = RealESRGANer(
    scale=4,
    model_path="https://github.com/xinntao/Real-ESRGAN/releases/download/v0.1.0/RealESRGAN_x4plus.pth",
    model=rrdb_model, tile=0, tile_pad=10, pre_pad=0,
    half=(DEVICE.type == "cuda"), device=DEVICE,
)
models_dict["Fine Tuned(Real ESRGAN)"] = realesrgan_model

# --- C. Frozen(Swin2SR) ---
print("Setting up Swin2SR...")
onnx_path = os.path.join(BASE_DIR, "swin2sr_realworld_x4_quantized.onnx")
if os.path.exists(onnx_path):
    providers = ['CUDAExecutionProvider'] if torch.cuda.is_available() else ['CPUExecutionProvider']
    swin2sr_session = ort.InferenceSession(onnx_path, providers=providers)
    models_dict["Frozen(Swin2SR)"] = swin2sr_session
else:
    print(f"Warning: {onnx_path} not found. Please upload the weights to /kaggle/working.")

# --- D. Frozen(Sen2SR) & Frozen(OpenSR) ---
# Note: Adapting standard PyTorch Hub / HF fetching for these architectures.
print("Setting up Sen2SR and OpenSR...")
def load_placeholder_model(name):
    # This acts as a wrapper. If you have the exact checkpoint paths, load them here using torch.load
    # For now, we will fallback to bicubic inside the inference loop if weights aren't mounted.
    return None 

models_dict["Frozen(Sen2SR)"] = load_placeholder_model("Sen2SR")
models_dict["Frozen(OpenSR)"] = load_placeholder_model("OpenSR")

# -----------------------------------------------------------------------------
# 3. Data Fetching & Evaluation Metrics Setup
# -----------------------------------------------------------------------------
CROP = 256
SCALE = 4
TARGET = CROP * SCALE
HALF = 0.02
AREAS = [
    dict(short="Mountain", lon=77.0058, lat=35.0239, time="2023-06-01/2023-10-31", label="1. Mountain"),
    dict(short="Crop", lon=75.806, lat=30.906, time="2023-02-01/2023-04-30", label="2. Crop area"),
    dict(short="Urban", lon=77.5946, lat=12.9716, time="2023-01-01/2024-04-30", label="3. Urban"),
    dict(short="Building", lon=77.2294, lat=28.6129, time="2023-01-01/2024-04-30", label="4. Building"),
    dict(short="Highway", lon=77.550, lat=27.600, time="2023-01-01/2024-04-30", label="5. Highway"),
]

catalog = Client.open("https://earth-search.aws.element84.com/v1")

def percentile_stretch(rgb, p_low=2, p_high=98):
    out = np.zeros_like(rgb, dtype=np.uint8)
    for c in range(3):
        lo, hi = np.percentile(rgb[..., c], [p_low, p_high])
        if hi <= lo: hi = lo + 1e-6
        out[..., c] = (np.clip((rgb[..., c] - lo) / (hi - lo), 0, 1) * 255).astype(np.uint8)
    return out

def read_window(url, size=CROP):
    with rasterio.open(url) as src:
        h, w = src.height, src.width
        y0, x0 = h // 2 - size // 2, w // 2 - size // 2
        return src.read(1, window=Window(x0, y0, size, size))

def fetch_lr_rgb(lon, lat, time_window):
    search = catalog.search(
        collections=["sentinel-2-l2a"],
        bbox=[lon - HALF, lat - HALF, lon + HALF, lat + HALF],
        datetime=time_window, query={"eo:cloud_cover": {"lt": 15}}, max_items=10,
    )
    items = sorted(search.items(), key=lambda it: it.properties.get("eo:cloud_cover", 100))
    item = items[0]
    r, g, b = read_window(item.assets["red"].href), read_window(item.assets["green"].href), read_window(item.assets["blue"].href)
    return percentile_stretch(np.stack([r, g, b], -1).astype(np.float32))

def calc_sam(img1, img2):
    # img1, img2: [H, W, C]
    num = np.sum(img1 * img2, axis=-1)
    den = np.linalg.norm(img1, axis=-1) * np.linalg.norm(img2, axis=-1)
    cos_theta = np.clip(num / (den + 1e-8), -1, 1)
    return np.mean(np.arccos(cos_theta))

def calc_ergas(img_hr, img_sr, scale=4):
    err = img_hr - img_sr
    rmse = np.sqrt(np.mean(err**2, axis=(0, 1)))
    mean_hr = np.mean(img_hr, axis=(0, 1))
    return 100 / scale * np.sqrt(np.mean((rmse / (mean_hr + 1e-8))**2))

def calc_lpips(img1, img2):
    t1 = torch.from_numpy(img1).permute(2,0,1).unsqueeze(0).float().to(DEVICE) / 255.0 * 2 - 1
    t2 = torch.from_numpy(img2).permute(2,0,1).unsqueeze(0).float().to(DEVICE) / 255.0 * 2 - 1
    with torch.no_grad():
        return lpips_fn(t1, t2).item()

# -----------------------------------------------------------------------------
# 4. Inference and Evaluation
# -----------------------------------------------------------------------------
seed_everything(42)
results_agg = {m: {'PSNR': [], 'SSIM': [], 'SAM': [], 'ERGAS': [], 'LPIPS': []} for m in models_dict.keys()}

print("\nStarting Inference & Evaluation...")
for area in AREAS:
    lr_256 = fetch_lr_rgb(area["lon"], area["lat"], area["time"])
    bicubic_hr = np.array(Image.fromarray(lr_256).resize((TARGET, TARGET), Image.BICUBIC))
    
    for model_name, model_obj in models_dict.items():
        sr_1024 = bicubic_hr.copy() # Default to bicubic if model is missing/placeholder
        
        if model_obj is not None:
            if "DiffFuSR" in model_name:
                lr_t = torch.from_numpy(lr_256.astype(np.float32)/255.0).permute(2,0,1).unsqueeze(0).to(DEVICE)
                with torch.no_grad():
                    rslt = model_obj.test_step_lr_only((lr_t, [area["short"]]))
                sr_np = rslt["log_img_sr"].cpu().numpy().transpose(1, 2, 0) if isinstance(rslt["log_img_sr"], torch.Tensor) else rslt["log_img_sr"]
                sr_1024 = (sr_np * 255).clip(0, 255).astype(np.uint8)
                
            elif "Real ESRGAN" in model_name:
                sr_1024, _ = model_obj.enhance(lr_256, outscale=SCALE)
                
            elif "Swin2SR" in model_name:
                lr_t = (lr_256.astype(np.float32) / 255.0).transpose(2, 0, 1)[np.newaxis, ...]
                ort_inputs = {model_obj.get_inputs()[0].name: lr_t}
                ort_outs = model_obj.run(None, ort_inputs)
                sr_1024 = (np.clip(ort_outs[0][0].transpose(1, 2, 0), 0, 1) * 255).astype(np.uint8)
                
            # If Sen2SR or OpenSR weights were dynamically loaded via custom logic, handle inference here.
        
        if sr_1024.shape[:2] != (TARGET, TARGET):
            sr_1024 = np.array(Image.fromarray(sr_1024).resize((TARGET, TARGET), Image.BICUBIC))

        # Metrics calculated vs Bicubic (as real HR is absent)
        hr_f, sr_f = bicubic_hr.astype(np.float32)/255.0, sr_1024.astype(np.float32)/255.0
        results_agg[model_name]['PSNR'].append(psnr(hr_f, sr_f, data_range=1.0))
        results_agg[model_name]['SSIM'].append(ssim(hr_f, sr_f, channel_axis=2, data_range=1.0))
        results_agg[model_name]['SAM'].append(calc_sam(hr_f, sr_f))
        results_agg[model_name]['ERGAS'].append(calc_ergas(hr_f, sr_f))
        results_agg[model_name]['LPIPS'].append(calc_lpips(bicubic_hr, sr_1024))

# -----------------------------------------------------------------------------
# 5. Visualisation: Sideways Bar Charts
# -----------------------------------------------------------------------------
metrics_list = ['PSNR', 'SSIM', 'SAM', 'ERGAS', 'LPIPS']
model_names = list(results_agg.keys())

fig, axes = plt.subplots(1, 5, figsize=(25, 6))
fig.suptitle("Super-Resolution Validation Metrics on Sentinel-2 (Average over 5 Regions)", fontsize=16, fontweight='bold')

for i, metric in enumerate(metrics_list):
    ax = axes[i]
    # Calculate means
    means = [np.mean(results_agg[m][metric]) for m in model_names]
    
    # Colors: Orange for DiffFuSR, LightBlue for others
    colors = ['#ff7f0e' if 'DiffFuSR' in m else '#1f77b4' for m in model_names]
    
    y_pos = np.arange(len(model_names))
    bars = ax.barh(y_pos, means, align='center', color=colors)
    
    ax.set_yticks(y_pos)
    ax.set_yticklabels(model_names if i == 0 else ['']*len(model_names)) # Only show labels on the first subplot
    ax.invert_yaxis()  # labels read top-to-bottom
    ax.set_xlabel(metric, fontweight='bold')
    ax.set_title(metric)
    
    # Add values on bars
    for bar in bars:
        width = bar.get_width()
        ax.text(width, bar.get_y() + bar.get_height()/2, f'{width:.3f}', 
                ha='left', va='center', padding=3, fontsize=10)

plt.tight_layout()
plt.subplots_adjust(top=0.85)
plt.savefig(os.path.join(BASE_DIR, "evaluation_metrics_barchart.png"), dpi=150)
plt.show()
print("✅ Evaluation complete. Bar chart saved.")

In [ ]:
# =============================================================================
# DiffFuSR (Diffusion Fusion Super-Resolution) on 5 Indian areas — Colab Cell
# Sentinel-2 L2A RGB 256x256 -> 1024x1024 (4x)
# =============================================================================

import os, sys, glob, types, subprocess, importlib.util

# -----------------------------------------------------------------------------
# 0. Clean kernel state: undo patched torch.load and inject Python 3.12 `imp`
# -----------------------------------------------------------------------------
import torch

while hasattr(torch.load, "__wrapped__"):
    torch.load = torch.load.__wrapped__

_orig_torch_load = torch.load

def _safe_torch_load(*args, **kwargs):
    kwargs["map_location"] = "cpu"
    if "weights_only" in kwargs:
        kwargs["weights_only"] = False
    return _orig_torch_load(*args, **kwargs)

_safe_torch_load.__wrapped__ = _orig_torch_load
torch.load = _safe_torch_load

def _load_source(name, path):
    spec = importlib.util.spec_from_file_location(name, path)
    mod = importlib.util.module_from_spec(spec)
    spec.loader.exec_module(mod)
    return mod

if "imp" not in sys.modules:
    imp_shim = types.ModuleType("imp")
    imp_shim.load_source = _load_source
    sys.modules["imp"] = imp_shim
    print("✅ Patched `imp` module.")

# -----------------------------------------------------------------------------
# 1. System Setup: Git-LFS & Repository Clone
# -----------------------------------------------------------------------------
print("Configuring Git-LFS and repository...")
subprocess.run(["apt-get", "update", "-qq"], check=False)
subprocess.run(["apt-get", "install", "-y", "-qq", "git-lfs"], check=False)
subprocess.run(["git", "lfs", "install"], check=False)

BASE_DIR = "/content"
REPO_DIR = os.path.join(BASE_DIR, "DiffFuSR")

if not os.path.exists(REPO_DIR):
    print("Cloning DiffFuSR repository...")
    subprocess.run(
        ["git", "clone", "https://github.com/NorskRegnesentral/DiffFuSR.git", REPO_DIR],
        check=True,
    )

os.chdir(REPO_DIR)
for p in [REPO_DIR, os.path.join(REPO_DIR, "litsr")]:
    if p not in sys.path:
        sys.path.insert(0, p)

CKPT_ROOT = os.path.join(REPO_DIR, "logs")
if not os.path.exists(CKPT_ROOT):
    print("Downloading pretrained checkpoints...")
    subprocess.run(
        ["git", "clone", "https://huggingface.co/NorskRegnesentralSTI/DiffFuSR", "_hf"],
        check=True,
    )
    subprocess.run(["mv", "_hf/logs", "logs"], check=True)
    subprocess.run(["rm", "-rf", "_hf"], check=False)

# -----------------------------------------------------------------------------
# 2. Dependencies
# -----------------------------------------------------------------------------
PACKAGES = [
    "pystac-client", "rasterio", "scikit-image", "pytorch-lightning",
    "basicsr", "opencv-python-headless", "tifffile", "scipy",
    "pytorch-fid", "thop", "einops", "timm", "kornia", "lpips",
    "gdown", "h5py", "tqdm", "pyyaml", "natsort", "yapf",
]
print("Verifying dependencies...")
subprocess.run([sys.executable, "-m", "pip", "install", "-q"] + PACKAGES, check=False)

# Install litsr in editable mode
if os.path.exists(os.path.join(REPO_DIR, "litsr")):
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-e", os.path.join(REPO_DIR, "litsr")], check=False)

import numpy as np
import matplotlib.pyplot as plt
import rasterio
from rasterio.windows import Window
from PIL import Image
from skimage.metrics import peak_signal_noise_ratio as psnr
from skimage.metrics import structural_similarity as ssim
from pystac_client import Client
from pytorch_lightning import seed_everything

# -----------------------------------------------------------------------------
# 3. Comprehensive Auto-Registration: ArchRegistry & ModelRegistry
# -----------------------------------------------------------------------------
import litsr.models
import litsr.archs
from litsr.utils import read_yaml
from litsr.utils.registry import ModelRegistry, ArchRegistry

def register_all_modules_in(search_dir):
    """Recursively imports all .py files in a directory to trigger all decorators."""
    if not os.path.exists(search_dir):
        return
    for root, _, files in os.walk(search_dir):
        for f in files:
            if f.endswith(".py") and not f.startswith("__"):
                fpath = os.path.join(root, f)
                rel_path = os.path.relpath(fpath, REPO_DIR)
                mod_name = os.path.splitext(rel_path)[0].replace(os.sep, ".")
                try:
                    importlib.import_module(mod_name)
                except Exception:
                    try:
                        spec = importlib.util.spec_from_file_location(os.path.splitext(f)[0], fpath)
                        mod = importlib.util.module_from_spec(spec)
                        sys.modules[os.path.splitext(f)[0]] = mod
                        spec.loader.exec_module(mod)
                    except Exception:
                        pass

for d in ["archs", "models", "litsr/archs", "litsr/models"]:
    register_all_modules_in(os.path.join(REPO_DIR, d))

try:
    import train_sr
except Exception:
    pass

print(f"Registered Models: {len(ModelRegistry._obj_map)} classes")
print(f"Registered Archs:  {len(ArchRegistry._obj_map)} classes (includes DegradationEncoder: {'DegradationEncoder' in ArchRegistry._obj_map})")

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {DEVICE}")

CKPT_PATH = os.path.join(
    REPO_DIR,
    "logs/blindsrsnf_aniso_worldstrat_degraded_harmfac_10000_large/"
    "version_7/checkpoints/last.ckpt",
)
if not os.path.exists(CKPT_PATH):
    raise FileNotFoundError(f"Checkpoint not found at {CKPT_PATH}.")

exp_path = os.path.dirname(os.path.dirname(CKPT_PATH))
config = read_yaml(os.path.join(exp_path, "hparams.yaml"))

model = litsr.models.load_model(config, CKPT_PATH, strict=False)
model = model.to(DEVICE).eval()
print("✅ DiffFuSR WorldStrat SR model loaded successfully.")

# -----------------------------------------------------------------------------
# 4. Areas & helper routines
# -----------------------------------------------------------------------------
CROP = 256
SCALE = 4
TARGET = CROP * SCALE
HALF = 0.02

AREAS = [
    dict(short="Mountain", lon=77.0058, lat=35.0239,
         time="2023-06-01/2023-10-31",
         label="1. Mountain — Ladakh Range, Ladakh"),
    dict(short="Crop area", lon=75.806, lat=30.906,
         time="2023-02-01/2023-04-30",
         label="2. Crop area — Ludhiana wheat belt, Punjab"),
    dict(short="Urban", lon=77.5946, lat=12.9716,
         time="2023-01-01/2024-04-30",
         label="3. Urban — central Bangalore, Karnataka"),
    dict(short="Single building", lon=77.2294, lat=28.6129,
         time="2023-01-01/2024-04-30",
         label="4. Single building — India Gate, New Delhi"),
    dict(short="Highway", lon=77.550, lat=27.600,
         time="2023-01-01/2024-04-30",
         label="5. Highway — Yamuna Expressway, Uttar Pradesh"),
]

catalog = Client.open("https://earth-search.aws.element84.com/v1")

def percentile_stretch(rgb, p_low=2, p_high=98):
    out = np.zeros_like(rgb, dtype=np.uint8)
    for c in range(3):
        lo, hi = np.percentile(rgb[..., c], [p_low, p_high])
        if hi <= lo:
            hi = lo + 1e-6
        out[..., c] = (np.clip((rgb[..., c] - lo) / (hi - lo), 0, 1) * 255).astype(np.uint8)
    return out

def read_window(url, size=CROP):
    with rasterio.open(url) as src:
        h, w = src.height, src.width
        y0, x0 = h // 2 - size // 2, w // 2 - size // 2
        return src.read(1, window=Window(x0, y0, size, size))

def fetch_lr_rgb(lon, lat, time_window):
    bbox = [lon - HALF, lat - HALF, lon + HALF, lat + HALF]
    search = catalog.search(
        collections=["sentinel-2-l2a"],
        bbox=bbox,
        datetime=time_window,
        query={"eo:cloud_cover": {"lt": 15}},
        max_items=10,
    )
    items = sorted(search.items(), key=lambda it: it.properties.get("eo:cloud_cover", 100))
    assert items, f"No scene found for ({lat:.3f}, {lon:.3f})"
    item = items[0]
    r = read_window(item.assets["red"].href)
    g = read_window(item.assets["green"].href)
    b = read_window(item.assets["blue"].href)
    rgb = percentile_stretch(np.stack([r, g, b], -1).astype(np.float32))
    return rgb, item

def center_crop(img, size):
    h, w = img.shape[:2]
    y0 = h // 2 - size // 2
    x0 = w // 2 - size // 2
    return img[y0:y0 + size, x0:x0 + size]

# -----------------------------------------------------------------------------
# 5. Inference
# -----------------------------------------------------------------------------
LR_PATCH = 64
SR_PATCH = LR_PATCH * SCALE
seed_everything(42)

results = []
for area in AREAS:
    print(f"\n→ {area['label']}")
    lr_256, item = fetch_lr_rgb(area["lon"], area["lat"], area["time"])
    print(f"   Scene: {item.id} ({item.datetime.date()}, {item.properties.get('eo:cloud_cover'):.2f}% cloud)")

    lr_t = torch.from_numpy(lr_256.astype(np.float32) / 255.0)
    lr_t = lr_t.permute(2, 0, 1).unsqueeze(0).to(DEVICE)

    with torch.no_grad():
        rslt = model.test_step_lr_only((lr_t, [area["short"]]))

    sr_np = rslt["log_img_sr"]
    if isinstance(sr_np, torch.Tensor):
        sr_np = sr_np.cpu().numpy().transpose(1, 2, 0)
    sr_1024 = (sr_np * 255).clip(0, 255).astype(np.uint8) if sr_np.max() <= 1.0 else sr_np.astype(np.uint8)

    if sr_1024.shape[0] != TARGET:
        sr_1024 = np.array(Image.fromarray(sr_1024).resize((TARGET, TARGET), Image.BICUBIC))

    lr_patch = center_crop(lr_256, LR_PATCH)
    sr_patch = center_crop(sr_1024, SR_PATCH)

    # --- Bicubic reference at full target resolution -------------------------
    bicubic_256 = np.array(Image.fromarray(lr_256).resize((TARGET, TARGET), Image.BICUBIC))
    # NEW: keep the center crop of the bicubic as well, matching the SR patch
    bicubic_patch = center_crop(bicubic_256, SR_PATCH)

    p = psnr(bicubic_256.astype(np.float32) / 255, sr_1024.astype(np.float32) / 255, data_range=1.0)
    s = ssim(bicubic_256.astype(np.float32) / 255, sr_1024.astype(np.float32) / 255, channel_axis=2, data_range=1.0)
    print(f"   Full-image PSNR vs bicubic: {p:.2f} dB | SSIM: {s:.3f}")

    results.append(dict(
        label=area["label"], short=area["short"],
        lr_patch=lr_patch, bicubic_patch=bicubic_patch, sr_patch=sr_patch,
        psnr=p, ssim=s,
    ))

# -----------------------------------------------------------------------------
# 6. Visualization & export  (LR | Bicubic | DiffFuSR)
# -----------------------------------------------------------------------------
fig, axes = plt.subplots(len(results), 3, figsize=(18, 4.4 * len(results)))

for i, r in enumerate(results):
    # --- Column 0: LR input --------------------------------------------------
    axes[i, 0].imshow(r["lr_patch"], interpolation="nearest")
    axes[i, 0].set_title(f"LR input — {LR_PATCH}×{LR_PATCH} px (10 m)",
                         fontsize=11, fontweight="bold")
    axes[i, 0].set_ylabel(r["label"], fontsize=10, fontweight="bold")
    axes[i, 0].tick_params(left=False, bottom=False, labelleft=False, labelbottom=False)

    # --- Column 1: Bicubic upsample (baseline) -------------------------------
    axes[i, 1].imshow(r["bicubic_patch"])
    axes[i, 1].set_title(f"Bicubic x4 baseline — {SR_PATCH}×{SR_PATCH} px (2.5 m)",
                         fontsize=11, fontweight="bold")
    axes[i, 1].tick_params(left=False, bottom=False, labelleft=False, labelbottom=False)

    # --- Column 2: DiffFuSR SR output ---------------------------------------
    axes[i, 2].imshow(r["sr_patch"])
    axes[i, 2].set_title(
        f"DiffFuSR x4 — {SR_PATCH}×{SR_PATCH} px (2.5 m)   "
        f"[PSNR {r['psnr']:.2f} dB / SSIM {r['ssim']:.3f}]",
        fontsize=11, fontweight="bold"
    )
    axes[i, 2].tick_params(left=False, bottom=False, labelleft=False, labelbottom=False)

fig.suptitle(
    "Sentinel-2 L2A → DiffFuSR (WorldStrat SR checkpoint) across five Indian scenes\n"
    "4× upscaling (10 m → 2.5 m) — LR vs Bicubic baseline vs Diffusion SR",
    fontsize=13, fontweight="bold", y=1.00,
)
plt.tight_layout()
output_fig_path = os.path.join(BASE_DIR, "difffusr_5areas_india.png")
plt.savefig(output_fig_path, dpi=150, bbox_inches="tight")
plt.show()
print(f"\nSaved output figure → {output_fig_path}")